In [0]:
# Environment selection as dropdown
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="fq_dev_pnl",
    choices=["fq_dev_pnl", "fq_test_pnl", "fq_prod_pnl"],
    label="Select environment"
)

# Source selection as combobox
dbutils.widgets.combobox(
    name="source",
    defaultValue="other",
    choices=["POSIST", "NETSUITE", "other"],
    label="Source"
)

# Domain selection as combobox
dbutils.widgets.combobox(
    name="domain",
    defaultValue="pnl_budget_flat_data",
    choices=["brand_ho_allocation_cost","pnl_budget_flat_data"],
    label="Domain"
)

environment = dbutils.widgets.get("environment")
source = dbutils.widgets.get("source")
domain = dbutils.widgets.get("domain")

staging = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `fq_dev_extloc_staging`"
).select("url").collect()[0][0]

checkpoint = 'abfss://fq-dev-pnl-container@fqadfstoragedev.dfs.core.windows.net/checkpoints/'

In [0]:
from pyspark.sql.functions import *

cdc_raw_data = spark.read.option('header', True).format('csv').load(f'{staging}/FoodQuest/P&L Flat data/P&L Budget Flat Data with Store HO 2026.csv').filter(col("Year")=="2026")
cdc_raw_data.filter(col("Column 1").like("%Allocation%")).display()
display(cdc_raw_data)

In [0]:
%sql CREATE EXTERNAL TABLE IF NOT EXISTS fq_dev_pnl_catalog.bronze.pnl_budget_flat_data
USING DELTA
LOCATION 'abfss://fq-dev-pnl-container@fqadfstoragedev.dfs.core.windows.net/bronze/pnl_budget_flat_data' 
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
import time
from pyspark.sql.functions import col, expr, current_timestamp, regexp_replace, lit, substring, substring_index, to_timestamp, size

bronzeDF = (spark.read
                .format("csv")
                .option('header', True)
                .option("inferSchema", "true")
                .load(f'{staging}/FoodQuest/P&L Flat data/budget_data_to_load.csv')
            )
display(bronzeDF)

(bronzeDF.withColumn("ingestion_ts", current_timestamp())
        .withColumn('file_name', substring_index(col('_metadata.file_name'), '.', 1))
        .withColumn('file_path', regexp_replace(col('_metadata.file_path'), '%20', ' '))
        .withColumn('sys_id', expr('uuid()'))
        .write
        .option("overwriteSchema", "true")
        .option("delta.columnMapping.mode", "name")
        .mode('overwrite')
        .saveAsTable(f"`{environment}_catalog`.`bronze`.`{domain}`", mergeSchema=True)
)



In [0]:
bronzeDF.filter(col("Column 1").like("%Allocation%")).display()

In [0]:
%sql
SELECT 
  COUNT(*) AS total_rows,
  count(distinct Year)
FROM fq_dev_pnl_catalog.bronze.pnl_budget_flat_data;

In [0]:
%sql select * from fq_dev_pnl_catalog.bronze.pnl_budget_flat_data  where `Column 1` in ('Store HO Allocation')